In [1]:
# Importing required Libraries
import os
import numpy as np
import pandas as pd
import random
import cv2
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.multioutput import MultiOutputClassifier
from sklearn import tree
from sklearn.decomposition import PCA
from PIL import Image


In [2]:
# loading the processed data file
df = pd.read_excel("../Processed_Data.xlsx")
df.head()

,Patient ID,Patient Age,Patient Sex,Filename,Diagnosis,N,D,G,C,A,H,M,O
0,0,69,Female,0_left.jpg,cataract,0,0,0,1,0,0,0,0
1,1,57,Male,1_left.jpg,normal fundus,1,0,0,0,0,0,0,0
2,2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy",0,1,0,0,0,0,0,1
3,3,66,Male,3_left.jpg,normal fundus,1,0,0,0,0,0,0,0
4,4,53,Male,4_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1


In [3]:
# Print the length of the DataFrame 'df' to display the number of rows in the dataset
print("Dataset length: ", len(df))

# Print the shape of the DataFrame 'df', which includes the number of rows and columns in a tuple
print("Dataset Shape: ", df.shape)

Dataset length:  7000
Dataset Shape:  (7000, 13)


In [4]:
# Creating a new traget columns 
def createTarget(df):
    # Create a new column 'Labels' in the DataFrame and initialize it with a list of 8 zeros for each row
    df['Labels'] = [[0, 0, 0, 0, 0, 0, 0, 0] for i in range(0, len(df))]

    # Define the target columns you want to extract values from
    target_columns = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

    # Create an empty list called 'label' to store values for each row
    label = []

    # Loop through each row in the DataFrame
    for i in range(0, len(df)):
        # Loop through the target columns
        for target in target_columns:
            # Append the value from the current target column for the current row to the 'label' list
            label.append(df.loc[i, target])

        # Update the 'Labels' column for the current row with the 'label' list
        df.at[i, 'Labels'] = label

        # Reset the 'label' list for the next row
        label = []

In [5]:
# Call the 'createTarget' function to modify the 'df' DataFrame
createTarget(df)
df.head(10)

,Patient ID,Patient Age,Patient Sex,Filename,Diagnosis,N,D,G,C,A,H,M,O,Labels
0,0,69,Female,0_left.jpg,cataract,0,0,0,1,0,0,0,0,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,1,57,Male,1_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy",0,1,0,0,0,0,0,1,"[0, 1, 0, 0, 0, 0, 0, 1]"
3,3,66,Male,3_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,4,53,Male,4_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1,"[0, 0, 0, 0, 0, 0, 0, 1]"
5,5,50,Female,5_left.jpg,moderate non proliferative retinopathy,0,1,0,0,0,0,0,0,"[0, 1, 0, 0, 0, 0, 0, 0]"
6,6,60,Male,6_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1,"[0, 0, 0, 0, 0, 0, 0, 1]"
7,7,60,Female,7_left.jpg,drusen,0,0,0,0,0,0,0,1,"[0, 0, 0, 0, 0, 0, 0, 1]"
8,8,59,Male,8_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
9,9,54,Male,9_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"


In [6]:
# Create a new DataFrame 'upd_df' by dropping specified columns from the original DataFrame 'df'
# The columns 'Patient ID', 'Patient Age', 'Patient Sex', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'Diagnosis'
# are removed from 'upd_df'
upd_df = df.drop(['Patient ID', 'Patient Age', 'Patient Sex', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'Diagnosis'], axis=1)

# 'upd_df' now contains the remaining columns after dropping the specified ones
upd_df


,Filename,Labels
0,0_left.jpg,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,1_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,2_left.jpg,"[0, 1, 0, 0, 0, 0, 0, 1]"
3,3_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,4_left.jpg,"[0, 0, 0, 0, 0, 0, 0, 1]"
...,...,...
6995,4686_right.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]"
6996,4688_right.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]"
6997,4689_right.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]"
6998,4690_right.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]"


In [7]:
# Path to your image dataset
image_dataset_path = "../ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training images"

In [8]:
# Initialize the lists to hold the feature vectors and labels
features = []
labels = []

# Process the images and labels
for index, row in df.iterrows():
    # Open the image
    image_path = os.path.join(image_dataset_path, row['Filename'])
    with Image.open(image_path) as img:
        img = img.resize((128, 128))  # Resize the image
        img_array = np.array(img)  # Convert to numpy array
        img_array = img_array.flatten()  # Flatten the array

    # Append the image (as a flattened array) to the features
    features.append(img_array)
    # Append the label to the labels list
    labels.append(row['Labels'])

# Convert lists to numpy arrays
features = np.array(features)
labels = np.array(labels)

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.3, random_state=42)

# Initialize the Decision Tree Classifier
clf = MultiOutputClassifier(DecisionTreeClassifier(random_state=42))

# Train the model
clf.fit(X_train, y_train)

# Predict the labels on the test set
y_pred = clf.predict(X_test)

In [12]:
# Calculate the accuracy of a machine learning model by comparing its predictions 'y_pred'
# to the true labels 'y_test' from the test set
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy of the Decision Tree classifier on the test set as a percentage with two decimal places
print(f'Accuracy of the Decision Tree classifier on the test set: {accuracy:.2%}')

Accuracy of the Decision Tree classifier on the test set: 15.81%


In [13]:
from sklearn.metrics import classification_report
classification_report = classification_report(y_pred, y_test)
print(classification_report)

              precision    recall  f1-score   support

           0       0.53      0.48      0.50      1049
           1       0.28      0.26      0.27       574
           2       0.15      0.13      0.14       113
           3       0.23      0.18      0.20       131
           4       0.08      0.06      0.06       127
           5       0.12      0.08      0.10        98
           6       0.48      0.40      0.44        98
           7       0.20      0.19      0.19       400

   micro avg       0.36      0.32      0.33      2590
   macro avg       0.26      0.22      0.24      2590
weighted avg       0.35      0.32      0.33      2590
 samples avg       0.37      0.27      0.29      2590



C:\Users\mimiw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


### FEATURE EXTRACTION 
Applying Feature extraction to the images to find out if we can increase the accuracy because the accuracy is very low. 

Applying PCA as our feature extraction technique.

In [14]:
# Initializing the PCA object with the specified number of components
pca = PCA(n_components=30, random_state=42)

# Fitting the PCA model to the data and transforming it to the reduced-dimensional space
reduced_features = pca.fit_transform(features)

reduced_features

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(reduced_features, labels, test_size=0.2, random_state=42)

# Initialize the Decision Tree Classifier
clf = MultiOutputClassifier(DecisionTreeClassifier(random_state=42))

# Train the model
clf.fit(X_train, y_train)

MultiOutputClassifier(estimator=DecisionTreeClassifier(random_state=42))

In [15]:
# Predict the labels on the test set
y_pred = clf.predict(X_test)
# Evaluate the model using accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy of the Decision Tree classifier on the test set: {accuracy:.2%}')

Accuracy of the Decision Tree classifier on the test set: 15.14%


In [18]:
from sklearn.metrics import classification_report
print(classification_report(y_pred, y_test))

              precision    recall  f1-score   support

           0       0.51      0.47      0.49       675
           1       0.30      0.28      0.28       385
           2       0.16      0.14      0.15        83
           3       0.24      0.25      0.24        73
           4       0.04      0.03      0.03        75
           5       0.14      0.11      0.12        57
           6       0.28      0.29      0.29        52
           7       0.16      0.17      0.16       248

   micro avg       0.34      0.32      0.33      1648
   macro avg       0.23      0.22      0.22      1648
weighted avg       0.34      0.32      0.33      1648
 samples avg       0.34      0.26      0.28      1648



C:\Users\mimiw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


#### RESULT

Using decision Tree on our images to predict the diagnosis gives a low accuracy percentage (15.81%).

We decided to use PCA feature extraction to check if we can increase the accuracy of the Decision Tree.
 
Using PCA we discovered that the accuracy of the Decision Tree dropped (15.14%).

This proves that using Decision Tree for image classifier is not a suitable algorithm for image classifier.
